# 01 - Fuentes de datos y construcción del dataset documental

## Objetivo del notebook

Este notebook tiene como objetivo construir una base documental fiable para el proyecto de Machine Learning sobre los documentos desclasificados del 23-F.

La pregunta principal que debe responder este notebook es:

> ¿Qué se puede analizar realmente con estos documentos?

Antes de aplicar modelos, necesitamos entender:

- qué fuentes de datos existen;
- qué información aporta cada fuente;
- qué documentos pueden inventariarse;
- qué textos completos pueden extraerse;
- qué PDFs pueden descargarse;
- qué metadatos están disponibles;
- qué limitaciones tiene el corpus;
- qué mini casos de uso son viables para la entrega final.

## Fuentes que se evaluarán

En esta fase se analizarán tres fuentes:

1. **[Buscador RTVE 23-F](https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/)**  
   Fuente principal indicada en el enunciado. Contiene título, páginas, resumen, palabras clave, texto completo y visor del documento original.

2. **[Lista íntegra RTVE de documentos desclasificados](https://www.rtve.es/noticias/20260225/todos-documentos-desclasificados-23f-lista-integra/16954095.shtml)**  
   Página complementaria de RTVE que permite localizar enlaces directos a documentos/PDFs.

3. **[Página oficial de La Moncloa sobre la desclasificación de documentos del 23-F](https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx)**  
   Fuente institucional de contraste. Se usará para verificar cobertura documental y detectar posibles documentos que no aparezcan en RTVE.

## Criterio de trabajo

No se parte de la idea de que una fuente sea automáticamente mejor que otra.  
Primero se compararán RTVE y La Moncloa en términos de cobertura, texto disponible, metadatos, facilidad de descarga, calidad del contenido y utilidad para análisis ML/NLP.

A partir de esa comparación se decidirá:

- qué fuente será la base principal del dataset;
- qué fuente se usará como respaldo o contraste;
- qué documentos se podrán analizar;
- qué mini casos son técnicamente viables.

In [1]:
# ============================================================
# 0. Configuración inicial del notebook
# ============================================================

from pathlib import Path
import os
import sys
import json
import re
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0.1. Localización robusta de la raíz del proyecto
# ------------------------------------------------------------
# Objetivo:
# Detectar automáticamente la carpeta raíz del repositorio,
# independientemente de si el notebook se ejecuta desde /notebooks
# o desde la raíz del proyecto.
#
# Criterio:
# Se busca una carpeta que contenga ".git" o, como respaldo,
# README.md y la carpeta notebooks/.

def find_project_root(start_path: Path) -> Path:
    """
    Busca la raíz del proyecto subiendo desde start_path.

    La raíz se identifica por:
    - presencia de carpeta .git, o
    - presencia simultánea de README.md y notebooks/

    Parameters
    ----------
    start_path : Path
        Ruta inicial desde la que se empieza a buscar.

    Returns
    -------
    Path
        Ruta raíz del proyecto.

    Raises
    ------
    FileNotFoundError
        Si no se encuentra una raíz válida.
    """
    start_path = start_path.resolve()

    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        has_git = (path / ".git").exists()
        has_readme_and_notebooks = (path / "README.md").exists() and (path / "notebooks").exists()

        if has_git or has_readme_and_notebooks:
            return path

    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Ejecuta el notebook desde dentro del repositorio."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# ------------------------------------------------------------
# 0.2. Definición de rutas relativas del proyecto
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_PDFS_RTVE_DIR = RAW_DIR / "pdfs_rtve"
RAW_PDFS_MONCLOA_DIR = RAW_DIR / "pdfs_moncloa"
RAW_TEXTS_RTVE_DIR = RAW_DIR / "texts_rtve"
RAW_TEXTS_MONCLOA_DIR = RAW_DIR / "texts_moncloa"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

SRC_DIR = PROJECT_ROOT / "src"

# Crear carpetas si no existen
for folder in [
    DATA_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    RAW_PDFS_RTVE_DIR,
    RAW_PDFS_MONCLOA_DIR,
    RAW_TEXTS_RTVE_DIR,
    RAW_TEXTS_MONCLOA_DIR,
    OUTPUTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Permitir importar módulos propios desde src/
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# ------------------------------------------------------------
# 0.3. URLs de trabajo
# ------------------------------------------------------------

URL_RTVE_BUSCADOR = "https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/"
URL_RTVE_LISTA = "https://www.rtve.es/noticias/20260225/todos-documentos-desclasificados-23f-lista-integra/16954095.shtml"
URL_MONCLOA = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx"

SOURCE_URLS = {
    "rtve_buscador": URL_RTVE_BUSCADOR,
    "rtve_lista": URL_RTVE_LISTA,
    "moncloa": URL_MONCLOA,
}

# ------------------------------------------------------------
# 0.4. Configuración visual de pandas
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# 0.5. Comprobación inicial
# ------------------------------------------------------------

print("Configuración inicial completada.")
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Carpeta de datos: {DATA_DIR}")
print(f"Carpeta interim: {INTERIM_DIR}")
print(f"Carpeta processed: {PROCESSED_DIR}")
print(f"Carpeta outputs: {OUTPUTS_DIR}")

Configuración inicial completada.
Raíz del proyecto: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML
Carpeta de datos: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data
Carpeta interim: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/interim
Carpeta processed: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/processed
Carpeta outputs: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/outputs
